In [6]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import gc
import warnings
warnings.filterwarnings('ignore')

print("✅ 导入完成")

✅ 导入完成


In [7]:
X = np.load('/root/X_interpolated.npy')
y = np.load('/root/y_interpolated.npy')

print(f"X 形状: {X.shape}")
print(f"y 分布: 0={sum(y==0)}, 1={sum(y==1)}")
print(f"X 中有 NaN: {np.isnan(X).any()}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")

X 形状: (14662, 4096, 23)
y 分布: 0=7595, 1=7067
X 中有 NaN: False
使用设备: cuda


In [8]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        self.attention = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(embed_dim)
    
    def forward(self, x):
        attn_output, _ = self.attention(x, x, x)
        attn_output = self.dropout(attn_output)
        return self.norm(x + attn_output)

class ConvMHSA(nn.Module):
    def __init__(self, input_channels=23, seq_len=4096, embed_dim=64, num_heads=8, num_layers=4, num_classes=2):
        super().__init__()
        
        self.conv_layers = nn.Sequential(
            nn.Conv1d(input_channels, embed_dim, kernel_size=8, stride=4, padding=2),
            nn.BatchNorm1d(embed_dim),
            nn.ReLU(),
            nn.Conv1d(embed_dim, embed_dim, kernel_size=8, stride=2, padding=3),
            nn.BatchNorm1d(embed_dim),
            nn.ReLU(),
        )
        self.compressed_len = 512
        
        self.pos_embedding = nn.Parameter(torch.randn(1, self.compressed_len, embed_dim) * 0.01)
        
        self.attn_layers = nn.ModuleList([
            MultiHeadSelfAttention(embed_dim, num_heads) for _ in range(num_layers)
        ])
        
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )
    
    def forward(self, x):
        x = self.conv_layers(x)
        x = x.permute(0, 2, 1)
        x = x + self.pos_embedding
        for attn in self.attn_layers:
            x = attn(x)
        x = x.mean(dim=1)
        return self.classifier(x)

print("✅ ConvMHSA 模型定义完成")

✅ ConvMHSA 模型定义完成


In [9]:
def train_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for X_batch, y_batch in dataloader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)

def evaluate(model, dataloader, device):
    model.eval()
    all_preds, all_probs, all_labels = [], [], []
    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch = X_batch.to(device)
            outputs = model(X_batch)
            probs = F.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())
            all_labels.extend(y_batch.numpy())
    return np.array(all_preds), np.array(all_probs), np.array(all_labels)

print("✅ 训练和评估函数定义完成")

✅ 训练和评估函数定义完成


In [10]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

accuracies, f1_scores, auc_scores = [], [], []

print("\n" + "="*60)
print("ConvMHSA 5折交叉验证（跑满50轮）")
print("="*60)

fold = 1
for train_idx, val_idx in skf.split(X, y):
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    print(f"\n--- Fold {fold} ---")
    print(f"  训练集: {len(train_idx):,} 样本")
    print(f"  验证集: {len(val_idx):,} 样本")
    
    X_train_t = torch.tensor(np.transpose(X_train, (0, 2, 1)), dtype=torch.float32)
    X_val_t = torch.tensor(np.transpose(X_val, (0, 2, 1)), dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.long)
    y_val_t = torch.tensor(y_val, dtype=torch.long)
    
    train_dataset = TensorDataset(X_train_t, y_train_t)
    val_dataset = TensorDataset(X_val_t, y_val_t)
    
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
    
    model = ConvMHSA().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-5)
    criterion = nn.CrossEntropyLoss()
    
    # 🔥 删除了 Early Stopping，固定跑 50 轮
    for epoch in range(50):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                val_loss += loss.item()
        val_loss /= len(val_loader)
        
        if epoch % 10 == 0:
            print(f"  Epoch {epoch}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")
    
    y_pred, y_prob, _ = evaluate(model, val_loader, device)
    
    acc = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    auc = roc_auc_score(y_val, y_prob)
    
    accuracies.append(acc)
    f1_scores.append(f1)
    auc_scores.append(auc)
    
    print(f"  准确率: {acc:.4f}, F1: {f1:.4f}, AUC: {auc:.4f}")
    fold += 1
    
    del model, X_train_t, X_val_t, y_train_t, y_val_t
    gc.collect()

print("\n" + "="*60)
print("📊 ConvMHSA 最终结果（跑满50轮）")
print("="*60)
print(f"  准确率: {np.mean(accuracies):.4f} ± {np.std(accuracies):.4f}")
print(f"  F1分数: {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")
print(f"  AUC:    {np.mean(auc_scores):.4f} ± {np.std(auc_scores):.4f}")
print("="*60)


ConvMHSA 5折交叉验证（跑满50轮）

--- Fold 1 ---
  训练集: 11,729 样本
  验证集: 2,933 样本
  Epoch 0: train_loss=0.6954, val_loss=0.6972
  Epoch 10: train_loss=0.6810, val_loss=0.6806
  Epoch 20: train_loss=0.6634, val_loss=0.7124
  Epoch 30: train_loss=0.6529, val_loss=0.6564
  Epoch 40: train_loss=0.6187, val_loss=0.8322
  准确率: 0.6601, F1: 0.5671, AUC: 0.7368

--- Fold 2 ---
  训练集: 11,729 样本
  验证集: 2,933 样本
  Epoch 0: train_loss=0.6955, val_loss=0.6891
  Epoch 10: train_loss=0.6854, val_loss=0.6839
  Epoch 20: train_loss=0.6670, val_loss=0.6949
  Epoch 30: train_loss=0.6585, val_loss=0.6884
  Epoch 40: train_loss=0.6471, val_loss=0.6461
  准确率: 0.5469, F1: 0.1497, AUC: 0.6445

--- Fold 3 ---
  训练集: 11,730 样本
  验证集: 2,932 样本
  Epoch 0: train_loss=0.6967, val_loss=0.6922
  Epoch 10: train_loss=0.6764, val_loss=0.6784
  Epoch 20: train_loss=0.6647, val_loss=0.7458
  Epoch 30: train_loss=0.6534, val_loss=0.6711
  Epoch 40: train_loss=0.6294, val_loss=0.6396
  准确率: 0.6514, F1: 0.5277, AUC: 0.7334

--- Fold 